# Bar data

The same notebook as [`../bar_data.ipynb`](../bar_data.ipynb), written against
[`ib_async`](https://github.com/ib-api-reloaded/ib_async) instead of the TWS API
shape. The library is unmodified and installed as usual; `ibx.ib_async.attach`
replaces the one layer of it that expects a socket to a gateway.

How far back the venue holds a series, the bars themselves, and a frame.

## Connecting

`IB.connect` was written for a gateway, so it takes a host, a port and a client
id. Here it takes none of them: the credentials go to `attach`, and there is no
local process to reach.

`ib.sleep()` rather than `time.sleep()` throughout. The library's loop runs on
this thread, and a plain sleep stops it — every stream then reads as dead.

In [ ]:
import os
from dotenv import load_dotenv
from ib_async import IB, util
import ibx.ib_async

util.startLoop()
load_dotenv()

ib = ibx.ib_async.attach(
    IB(),
    username=os.environ["IB_USERNAME"],
    password=os.environ["IB_PASSWORD"],
    paper=True,
)
ib.connect()          # names no host: there is no gateway to name

print(f"connected: {ib.isConnected()}")
print(f"accounts:  {ib.managedAccounts()}")

## How far back it goes

In [ ]:
from ib_async import Stock

spy = Stock("SPY", "SMART", "USD")
ib.qualifyContracts(spy)

head = ib.reqHeadTimeStamp(spy, whatToShow="TRADES", useRTH=True)
print(f"earliest bar: {head}")

## Historical bars

In [ ]:
bars = ib.reqHistoricalData(
    spy,
    endDateTime="",
    durationStr="5 D",
    barSizeSetting="1 hour",
    whatToShow="TRADES",
    useRTH=True,
)
print(f"{len(bars)} bars")
for b in bars[-5:]:
    print(f"{b.date}  o {b.open:8.2f}  h {b.high:8.2f}  l {b.low:8.2f}  c {b.close:8.2f}  vol {b.volume}")

## As a frame

`util.df` is the library's own conversion, and it takes these bars as they
arrive.

In [ ]:
df = util.df(bars)
df.tail()

## Keeping it up to date

With `keepUpToDate`, the last bar is amended as it forms rather than
repeated.

In [ ]:
live = ib.reqHistoricalData(
    spy,
    endDateTime="",
    durationStr="1 D",
    barSizeSetting="5 mins",
    whatToShow="TRADES",
    useRTH=False,
    keepUpToDate=True,
)
ib.sleep(10)
print(f"{len(live)} bars, last: {live[-1]}")
ib.cancelHistoricalData(live)

In [ ]:
ib.disconnect()